# Sistema multi-agente Strands com Branching de Memória do AgentCore

## Introdução

Este notebook demonstra como implementar um **sistema multi-agente com branching de memória** utilizando o AWS AgentCore Memory e o framework Strands. Este exemplo apresenta um recurso avançado de memória: **branching de conversas**, que permite que agentes façam fork do histórico de conversas e explorem caminhos alternativos de conversa, preservando a thread original da conversa.

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial    | Conversa de Curto Prazo com Branching de Memória                                 |
| Caso de uso do agente | Assistente de Planejamento de Viagens                                          |
| Framework de agentes | Strands Agents                                                                  |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | AgentCore Short-term Memory, Strands Agents, Recuperação de memória via Tool |
| Complexidade do exemplo | Iniciante                                                                    |


O que você aprenderá:

- Como implementar branching de memória para fazer fork do histórico de conversas
- Criar agentes especializados que trabalham em diferentes branches de conversa
- Gerenciar o ciclo de vida de branches (criação, inicialização e alternância entre branches)
- Manter o contexto da conversa tanto nas conversas principais quanto nas ramificadas

### Contexto do cenário

Neste exemplo, criaremos um **Sistema de Planejamento de Viagens** que demonstra o branching de memória:
1. Uma conversa no **branch principal** onde todas as conversas são armazenadas
2. Um **branch do agente de voos** que faz fork da conversa principal para explorar opções de voos
3. Um **branch do agente de hotéis** que faz fork da conversa principal para explorar opções de hotéis
4. Ambos os agentes podem acessar o histórico de conversas compartilhado do branch principal

Esta abordagem demonstra como o branching de memória possibilita:
- **Exploração paralela**: Agentes podem explorar cenários "e se" sem afetar a conversa principal
- **Preservação de contexto**: Conversas ramificadas mantêm acesso ao histórico original da conversa
- **Fluxos de trabalho especializados**: Cada branch pode seguir seu próprio caminho de conversa, mantendo-se fundamentado no contexto original

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Pré-requisitos
- Python 3.10+
- Conta AWS com permissões apropriadas
- IAM role da AWS com permissões apropriadas para o AgentCore Memory
- Acesso aos modelos do Amazon Bedrock

Vamos começar configurando nosso ambiente e criando nosso recurso de memória compartilhada!

## Passo 1: Configuração do ambiente
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para que este notebook funcione.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

Defina a região e a role com as permissões apropriadas para os modelos do Amazon Bedrock e o AgentCore

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## Passo 2: Criando Memória Compartilhada
Nesta seção, criaremos um recurso de memória que será compartilhado entre nossos agentes especializados.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None


In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Memory...")
    memory_name = memory_name

    # Create the memory resource
    memory = client.create_memory_and_wait(
        name=memory_name,                       # Unique name for this memory store
        description="Travel Agent STM",         # Human-readable description
        strategies=[],                          # No special memory strategies for short-term memory
        event_expiry_days=7,                    # Memories expire after 7 days
        max_wait=300,                           # Maximum time to wait for memory creation (5 minutes)
        poll_interval=10                        # Check status every 10 seconds
    )

    # Extract and print the memory ID
    memory_id = memory['id']
    print(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    print(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Entendendo a Memória com Suporte a Branching

O recurso de memória que criamos suporta **branching de conversas**, que possibilita:

1. **Sessão Compartilhada, Múltiplos Branches**: Todos os agentes utilizam o mesmo `memory_id`, `actor_id` e `session_id`, mas valores diferentes de `branch_name`
2. **Herança de Contexto**: Quando um branch é criado via fork, ele herda o histórico de conversas do branch pai (geralmente "main")
3. **Evolução Isolada**: Após o fork, cada branch mantém sua própria thread de conversa independente
4. **Recuperação Específica por Branch**: Agentes podem recuperar memórias do seu branch específico

Esta abordagem de branching permite que agentes especializados mantenham seus próprios contextos de conversa, permanecendo fundamentados no histórico de sessão compartilhado.

## Passo 3: Criar Memory Hook Provider com Suporte a Branching

Este passo define nossa classe customizada `ShortTermMemoryHook` que implementa o **branching de memória**. Este hook provider avançado estende as operações básicas de memória com capacidades de gerenciamento de branches:

### Principais Funcionalidades:
1. **Gerenciamento de Branches**: Cria e inicializa automaticamente branches de conversas
2. **Recuperação de Memórias**: Busca o histórico de conversas do branch especificado
3. **Salvamento de Memórias**: Armazena novas conversas no branch apropriado
4. **Fork de Branches**: Cria novos branches a partir da thread principal de conversas

### Como o Branching Funciona:
- Cada agente pode especificar um `branch_name` (padrão é "main")
- Branches não-main são automaticamente criados via fork da conversa principal
- Branches herdam o histórico de conversas até o ponto do fork
- Cada branch mantém seu próprio fluxo de conversa independente após o fork



In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory import MemorySessionManager
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_id: str, region_name: str = "us-west-2", branch_name: str = "main"):
        """Initialize the hook with a MemorySessionManager.

        Args:
            memory_id: The AgentCore Memory ID
            region_name: AWS region for the memory service
            branch_name: Branch name for this agent's memory (default: "main")
        """
        self.memory_manager = MemorySessionManager(
            memory_id=memory_id,
            region_name=region_name
        )
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # Cache session objects per actor/session combo
        self._branch_initialized = False  # Track if branch has been created

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """Get or create a MemorySession for the given actor/session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(
                actor_id=actor_id,
                session_id=session_id
            )
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """Initialize a branch if it doesn't exist and this is not the main branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
        """
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Check if branch already exists
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # Get the last event from main branch to fork from
                main_events = memory_session.list_events(branch_name="main")
                if main_events:
                    last_event = main_events[-1]
                    # Create the branch with an initial message
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(f"Starting {self.branch_name} branch", MessageRole.ASSISTANT)
                        ]
                    )
                    logger.info(f"✅ Created branch: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"Failed to initialize branch {self.branch_name}: {e}", exc_info=True)

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Initialize branch if needed (for non-main branches)
            if self.branch_name !="main":
                self._initialize_branch(actor_id, session_id)

            # Get the memory session
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Get last 5 conversation turns from this branch
            recent_turns = memory_session.get_last_k_turns(
                k=5,
                branch_name=self.branch_name
            )

            if recent_turns:
                # Format conversation history for context
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.content.get('role', 'unknown').lower()
                        text = message.content.get('content', {}).get('text', '')
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages)
                    logger.info(f"Loaded context from branch '{self.branch_name}' ({len(context_messages)} messages)")

                    # Add context to agent's system prompt
                    event.agent.system_prompt += (
                        f"\n\nRecent conversation history (from {self.branch_name}):\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )

                    logger.info(f"✅ Loaded {len(recent_turns)} recent conversation turns from branch '{self.branch_name}'")
            else:
                logger.info(f"No previous conversation history found in branch '{self.branch_name}'")

        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}", exc_info=True)

    def on_message_added(self, event: MessageAddedEvent):
        """Store conversation turns in memory on the appropriate branch"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Get the memory session
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Get the last message
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("Skipping empty message")
                return

            # Map role string to MessageRole enum
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # Store the message on the appropriate branch
            if self.branch_name == "main":
                # Main branch - just add turns normally
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)]
                )
            else:
                # Non-main branch - need to append to existing branch
                # Initialize branch if it doesn't exist
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # Get the latest event from this branch
                branch_events = memory_session.list_events(branch_name=self.branch_name)
                if branch_events:
                    # Add to existing branch by specifying branch name (without rootEventId)
                    memory_session.add_turns(
                        messages=[ConversationalMessage(content_text, message_role)],
                        branch={"name": self.branch_name}
                    )
                else:
                    # This shouldn't happen if _initialize_branch worked, but handle it
                    logger.warning(f"Branch {self.branch_name} not found after initialization")
                    self._initialize_branch(actor_id, session_id)

            logger.debug(f"✅ Stored message in branch '{self.branch_name}': {role_str}")

        except Exception as e:
            logger.error(f"Failed to store message: {e}", exc_info=True)

    def create_branch(self, actor_id: str, session_id: str,
                      root_event_id: str, branch_name: str,
                      messages: list):
        """Create a new conversation branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
            root_event_id: Event ID to branch from
            branch_name: Name for the new branch
            messages: List of ConversationalMessage objects to add to the branch
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.fork_conversation(
            root_event_id=root_event_id,
            branch_name=branch_name,
            messages=messages
        )

    def list_branches(self, actor_id: str, session_id: str):
        """List all branches for a session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            List of branch information
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.list_branches()

    def get_session(self, actor_id: str, session_id: str):
        """Get the memory session object for direct access.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks with the registry.

        Args:
            registry: The HookRegistry to register callbacks with
        """
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## Passo 4: Criar Arquitetura Multi-Agente com Branching de Memória

Nesta seção, criaremos agentes especializados que utilizam **diferentes branches de memória** para demonstrar a capacidade de branching:

### Estratégia de Branching:
- **Branch Principal**: Armazena a conversa do coordenador e serve como a thread base de conversa
- **Branch flight_agent_memory**: Um branch separado para conversas específicas sobre voos
- **Branch hotel_agent_memory**: Um branch separado para conversas específicas sobre hotéis

Cada agente especializado opera em seu próprio branch, que é automaticamente criado via fork da conversa principal quando usado pela primeira vez. Isso permite:

- Fluxos de conversa independentes para diferentes especializações
- Isolamento de contexto específico de domínio
- Preservação da thread principal de conversa

In [ ]:
# Import the necessary components
from strands import Agent, tool

In [ ]:
# Create unique actor IDs for each specialized agent but share the session ID
actor_id = f"travel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"travel/{actor_id}/preferences/"

### Criando Agentes Especializados com Memória Ramificada

A seguir, definiremos os system prompts e criaremos agentes que utilizam diferentes branches de memória. Observe como usamos o mesmo `actor_id` e `session_id`, mas valores diferentes de `branch_name` para criar contextos de conversa isolados:

In [ ]:
# System prompt for the hotel booking specialist
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# System prompt for the flight booking specialist
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

In [ ]:
flight_memory_hooks = None
hotel_memory_hooks = None

### Implementando Ferramentas de Agentes com Memória Específica por Branch

Agora implementaremos nossos agentes especializados como ferramentas. Cada agente recebe seu próprio memory hook configurado com um nome de branch específico:
- O assistente de voos utiliza o branch `flight_agent_memory`
- O assistente de hotéis utiliza o branch `hotel_agent_memory`

Quando esses agentes são invocados:
1. O hook verifica se o branch existe
2. Se não existir, ele faz fork de um novo branch a partir da conversa principal
3. A conversa do agente é armazenada em seu branch dedicado
4. O agente ainda pode acessar o contexto do branch principal 

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    global flight_memory_hooks
    try:
        if flight_memory_hooks is None:
            # Create hook with branch name "flight_agent_memory"
            flight_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="flight_agent_memory"
            )

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    global hotel_memory_hooks
    try:
        if hotel_memory_hooks is None:
            # Create hook with branch name "hotel_agent_memory"
            hotel_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="hotel_agent_memory"
            )

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Criando o Agente Coordenador (Branch Principal)

O agente coordenador opera no **branch principal** e delega para agentes especializados. Observe que:

- Quando ele chama os assistentes de voos ou hotéis, esses agentes criam seus próprios branches via fork
- O branch de cada agente especializado começa a partir do estado atual da conversa principal

In [ ]:
# System prompt for the coordinator agent
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
agent_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
            )

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    hooks=[agent_memory_hooks],
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
    state={
        "actor_id": actor_id,
        "session_id": session_id
    }
)

#### Seu Sistema Multi-Agente está pronto!!

## Vamos testar o Agente.

Vamos testar nosso sistema multi-agente com um cenário de planejamento de viagem:

In [ ]:
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
response = travel_agent("I would only like to focus on the flight at the moment. Direct flight with British Airways")

In [ ]:
print("\n=== Viewing Memory Branches ===")

if flight_memory_hooks or hotel_memory_hooks:
    # Get any memory session to list branches (they all point to the same session)
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # List all branches in the session
        branches = memory_session.list_branches()
        print(f"\n📊 Session has {len(branches)} branches total:")
        for branch in branches:
            print(f"  - Branch: {branch.name}")
            print(f"    └─ Events: {len(memory_session.list_events(branch_name=branch.name))}")
            print(f"    └─ Created: {branch.created}")

        print("\n💡 Each branch represents a different agent's memory:")
        print("  • 'main' = Travel coordinator conversations")
        print("  • 'flight_agent_memory' = Flight assistant conversations")
        print("  • 'hotel_agent_memory' = Hotel assistant conversations")

In [ ]:
print("\n=== Accessing Branch-Specific Events ===")

if flight_memory_hooks or hotel_memory_hooks:
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # Get events from the main branch (coordinator)
        main_events = memory_session.list_events(branch_name="main")
        print(f"\n🌳 Main Branch - Coordinator ({len(main_events)} events):")
        if main_events:
            for event in main_events[-3:]:  # Show last 3 events
                for payload in event.payload:
                    if 'conversational' in payload:
                        role = payload['conversational']['role']
                        text = payload['conversational']['content']['text']
                        print(f"  {role}: {text[:100]}...")
        else:
            print("  No events found in main branch")

        # Get events from the flight agent branch
        try:
            flight_branch_events = memory_session.list_events(branch_name="flight_agent_memory")
            print(f"\n✈️  Flight Agent Branch ({len(flight_branch_events)} events):")
            if flight_branch_events:
                print("All flight-related conversations are stored here:")
                for event in flight_branch_events[-3:]:  # Show last 3 events
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - flight assistant wasn't called yet")
        except Exception as e:
            print(f"  Flight branch not created yet: {e}")

        # Get events from the hotel agent branch
        try:
            hotel_branch_events = memory_session.list_events(branch_name="hotel_agent_memory")
            print(f"\n🏨 Hotel Agent Branch ({len(hotel_branch_events)} events):")
            if hotel_branch_events:
                print("All hotel-related conversations are stored here:")
                for event in hotel_branch_events[-3:]:  # Show last 3 events
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - hotel assistant wasn't called yet")
        except Exception as e:
            print(f"  Hotel branch not created yet: {e}")

## Resumo

Neste notebook, demonstramos:

1. **Fundamentos do Branching de Memória**: Como criar e gerenciar branches de conversas no AgentCore Memory
2. **Agentes Específicos por Branch**: Como implementar agentes especializados que operam em diferentes branches de memória
3. **Criação Automática de Branches**: Como branches são automaticamente criados via fork da conversa principal quando usados pela primeira vez
4. **Persistência de Branches**: Como cada branch mantém seu próprio histórico de conversas independente
5. **Herança de Contexto**: Como conversas ramificadas herdam o contexto do branch principal até o ponto do fork

### Principais Benefícios do Branching de Memória:
- **Isolamento**: Cada agente mantém seu próprio contexto de conversa sem interferir nos outros
- **Flexibilidade**: Explore caminhos alternativos de conversa sem afetar a thread principal
- **Organização**: Mantenha conversas específicas de domínio organizadas em branches separados
- **Persistência**: Memórias específicas de branch persistem entre instâncias de agentes

Esta arquitetura de branching de memória fornece uma abordagem poderosa para construir sistemas multi-agente sofisticados onde diferentes agentes precisam manter contextos de conversa separados, porém relacionados.

## Limpeza
Vamos excluir a memória para limpar os recursos utilizados neste notebook.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)